# CMA-ES Optimizer for the LVG Local Volatility Model
**CMU Quant Finance Research Team**

---

## Overview

The **Local Variance Gamma (LVG)** model represents European call prices $C(K)$ via a piecewise-exponential time-value function $V(K)$:

$$C(K) = V(K) + (S_0 - K)^+$$

The model is calibrated by choosing:
- **Partition points** $\nu_j$: breakpoints between intervals on $[0, \bar{K}]$
- **Local volatilities** $\sigma_j > 0$: exponential decay rates on each interval

### Objective

The risk-neutral density is proportional to $C''(K)$. On each interval $[\nu_j, \nu_{j+1}]$, the ODE $V'' = \frac{1}{\sigma_j^2} V$ gives $C''(K) = \frac{V(K)}{\sigma_j^2}$.

At each partition point, $C''$ can jump. We define the smoothness penalty:

$$J(\theta) = \sum_j \left[ C''(\nu_j^+) - C''(\nu_j^-) \right]^2$$

We minimize $J$ using **CMA-ES** (Covariance Matrix Adaptation Evolution Strategy), a derivative-free optimizer suited for high-dimensional non-convex problems.

### Parameter Vector Layout

$$\theta = \underbrace{[\nu_1^L, \ldots, \nu_{R_1}^L]}_{\text{left partition}} \; \| \; \underbrace{[\tilde{\sigma}_1^L, \ldots, \tilde{\sigma}_{R_1}^L]}_{\text{raw left vols}} \; \| \; \underbrace{[\nu_1^R, \ldots, \nu_{R_2}^R]}_{\text{right partition}} \; \| \; \underbrace{[\tilde{\sigma}_1^R, \ldots, \tilde{\sigma}_{R_2}^R]}_{\text{raw right vols}}$$

Total dimension: $2R_1 + 2R_2 = 208$


## Imports

In [1]:
import numpy as np
import csv
import matplotlib
%matplotlib inline
import matplotlib.pyplot as plt

## Section 1: Softplus Transform

Local volatilities must be strictly positive. We store them in *raw* form $\tilde{\sigma} \in \mathbb{R}$ and recover $\sigma > 0$ via:

$$\sigma = \text{softplus}(\tilde{\sigma}) + 10^{-6}, \qquad \text{softplus}(x) = \log(1 + e^x)$$

The inverse is:

$$\tilde{\sigma} = \text{softplus}^{-1}(\sigma) = \log(e^\sigma - 1)$$

This matches the teammates' PyTorch implementation exactly. The $+10^{-6}$ floor ensures $\sigma$ is never exactly zero.


In [2]:
def softplus(x):
    """
    Numerically stable softplus: log(1 + exp(x)).
    Clips input to [-500, 20] before computing to avoid overflow,
    and uses the identity softplus(x) ≈ x for x > 20.
    """
    return np.where(x > 20, x, np.log1p(np.exp(np.clip(x, -500, 20))))


def inv_softplus(x):
    """
    Inverse softplus: log(exp(x) - 1).
    Converts an actual sigma value back to raw form so that
    softplus(raw) + 1e-6 recovers the original sigma.
    """
    return np.where(x > 20, x, np.log(np.expm1(np.clip(x, 1e-8, 500))))

## Section 2: Core Coefficient Computation

On each interval $[\nu_j, \nu_{j+1}]$, the time-value satisfies the ODE:

$$V''(K) = \frac{1}{\sigma_j^2} V(K)$$

whose general solution is:

$$V(K) = A_j e^{K/\sigma_j} + B_j e^{-K/\sigma_j}$$

### Numerical Stability: Distance-Scaled Coefficients

For strikes $K \sim 1200$, storing $A_j e^{K/\sigma_j}$ directly causes overflow. Instead we store coefficients scaled relative to each interval's **right** endpoint:

$$\tilde{A}_j = A_j \, e^{-d_j/\sigma_j}, \qquad \tilde{B}_j = B_j \, e^{+d_j/\sigma_j}$$

where $d_j = \nu_{j+1} - \nu_j$. Then $V(\nu_{j+1}) = \tilde{A}_j + \tilde{B}_j$ exactly, with no large exponentials.

### $C^1$ Matching Recursion

At each internal node $\nu_{j+1}$, continuity of $V$ and $V'$ gives:

$$\tilde{A}_{j+1} = \frac{1}{2}(V_j^R - \sigma_{j+1} V_j^{R\prime}) \, e^{-d_{j+1}/\sigma_{j+1}}$$
$$\tilde{B}_{j+1} = \frac{1}{2}(V_j^R + \sigma_{j+1} V_j^{R\prime}) \, e^{+d_{j+1}/\sigma_{j+1}}$$

### Solving for $\lambda_1, \lambda_2$ at $S_0$

The full call price is:

$$C(K) = \begin{cases} \lambda_1 V_L(K) + (S_0 - K) & K \leq S_0 \\ \lambda_2 V_R(K) & K > S_0 \end{cases}$$

$C^1$ continuity at $S_0$ gives the $2 \times 2$ linear system:

$$\lambda_1 V_L(S_0) = \lambda_2 V_R(S_0)$$
$$\lambda_1 V_L'(S_0) - \lambda_2 V_R'(S_0) = 1$$

where the $+1$ on the right comes from the kink in $(S_0 - K)^+$.


In [3]:
def _compute_wing_coefficients(theta, R1, R2, S0):
    """
    Compute all LVG model coefficients from a parameter vector theta.

    This is the shared computation used by calculate_J, compute_model_calls,
    and the plotting functions. Factored out to avoid code duplication.

    Parameters
    ----------
    theta : ndarray (2*R1 + 2*R2,)
    R1, R2 : int   number of left/right partition points
    S0 : float     spot price (junction between left and right wings)

    Returns
    -------
    coeff_left  : ndarray (R1, 2)  distance-scaled coefficients, left wing
    coeff_right : ndarray (R2, 2)  distance-scaled coefficients, right wing
    lambda1     : float  left wing amplitude (from C1-matching at S0)
    lambda2     : float  right wing amplitude (from C1-matching at S0)
    left_nodes  : ndarray (R1+1,)  partition points with S0 appended
    right_nodes : ndarray (R2+1,)  partition points with S0 prepended
    local_vols_left  : ndarray (R1,)  actual sigma values, left wing
    local_vols_right : ndarray (R2,)  actual sigma values, right wing
    """

    # ── Unpack theta ──────────────────────────────────────────────────────
    partition_pts_left   = theta[:R1]
    local_vols_left_raw  = theta[R1 : 2*R1]
    partition_pts_right  = theta[2*R1 : 2*R1 + R2]
    local_vols_right_raw = theta[2*R1 + R2 :]

    # S0 serves as the shared boundary between left and right wings:
    #   left wing  covers [0, S0]:   nodes = partition_pts_left  + [S0]
    #   right wing covers [S0, Kbar]: nodes = [S0] + partition_pts_right
    left_nodes  = np.append(partition_pts_left,   S0)   # length R1+1
    right_nodes = np.insert(partition_pts_right, 0, S0)  # length R2+1

    # Convert raw values to actual positive local volatilities
    local_vols_left  = softplus(local_vols_left_raw)  + 1e-6  # shape (R1,)
    local_vols_right = softplus(local_vols_right_raw) + 1e-6  # shape (R2,)

    # ── Left wing: propagate L -> R with unit scale (lambda1 = 1) ────────
    #
    # Boundary condition at K = 0 (left edge):
    #   V(0) = 0  (time value is zero at zero strike)
    #   V'(0) > 0 (time value increases as strike increases from 0)
    #
    # These give the unit starting coefficients for interval 0:
    #   coeff_left[0,0] = -exp(-dist_0 / sigma_0)
    #   coeff_left[0,1] = +exp(+dist_0 / sigma_0)
    # (One can verify: V(0) = coeff[0]*exp(0) + coeff[1]*exp(0) = 0 + ... wait)
    # Actually V at LEFT endpoint of interval 0 means K=left_nodes[0]=0:
    #   V(left_nodes[0]) = cd1*exp(0) + cd2*exp(0) = cd1 + cd2
    # With cd1 = coeff[0,0]*exp(dist/sig) = -1 and cd2 = coeff[0,1]*exp(-dist/sig) = 1
    # So V(0) = -1 + 1 = 0 ✓

    coeff_left    = np.zeros((R1, 2))
    dist_first    = left_nodes[1] - left_nodes[0]
    sigma_first   = local_vols_left[0]
    coeff_left[0, 0] = -np.exp(-dist_first / sigma_first)
    coeff_left[0, 1] =  np.exp( dist_first / sigma_first)

    # Propagate left to right: at each partition point nu_{j+1},
    # enforce C1 continuity (V and V' match across intervals)
    for j in range(R1 - 1):
        # V and V' at the RIGHT endpoint of interval j
        # (= left endpoint of interval j+1 = nu_{j+1})
        V_right_j  = coeff_left[j, 0] + coeff_left[j, 1]
        Vp_right_j = (1.0 / local_vols_left[j]) * (
                         -coeff_left[j, 0] + coeff_left[j, 1])

        # Set distance-scaled coefficients for interval j+1
        # using the C1 matching conditions
        dist_next  = left_nodes[j+2] - left_nodes[j+1]
        sigma_next = local_vols_left[j+1]

        coeff_left[j+1, 0] = 0.5 * (V_right_j - sigma_next * Vp_right_j) \
                              * np.exp(-dist_next / sigma_next)
        coeff_left[j+1, 1] = 0.5 * (V_right_j + sigma_next * Vp_right_j) \
                              * np.exp( dist_next / sigma_next)

    # ── Right wing: propagate R -> L with unit scale (lambda2 = 1) ───────
    #
    # Boundary condition at K = Kbar (right edge):
    #   V(Kbar) = 0  (call price = intrinsic = 0 at Kbar, far OTM)
    #   V'(Kbar) < 0 (time value decreases as strike approaches Kbar)

    coeff_right    = np.zeros((R2, 2))
    dist_last      = right_nodes[-1] - right_nodes[-2]
    sigma_last     = local_vols_right[-1]
    coeff_right[-1, 0] =  np.exp( dist_last / sigma_last)
    coeff_right[-1, 1] = -np.exp(-dist_last / sigma_last)

    # Propagate right to left: enforce C1 continuity at each partition point
    for j in range(R2 - 1, 0, -1):
        V_left_j  = coeff_right[j, 0] + coeff_right[j, 1]
        Vp_left_j = (1.0 / local_vols_right[j]) * (
                        -coeff_right[j, 0] + coeff_right[j, 1])

        dist_prev  = right_nodes[j] - right_nodes[j-1]
        sigma_prev = local_vols_right[j-1]

        coeff_right[j-1, 0] = 0.5 * (V_left_j - sigma_prev * Vp_left_j) \
                               * np.exp( dist_prev / sigma_prev)
        coeff_right[j-1, 1] = 0.5 * (V_left_j + sigma_prev * Vp_left_j) \
                               * np.exp(-dist_prev / sigma_prev)

    # ── Solve for lambda1 and lambda2 at S0 ──────────────────────────────
    #
    # The full call price is: C(K) = lambda1*V_left(K) + (S0-K)+   for K < S0
    #                                C(K) = lambda2*V_right(K)       for K > S0
    #
    # At K = S0, we need C1 continuity of C(K):
    #   Value:      lambda1 * V_left(S0) = lambda2 * V_right(S0)
    #   Derivative: lambda1 * V'_left(S0) - lambda2 * V'_right(S0) = 1
    #               (the +1 comes from the -1 kink in the derivative of (S0-K)+)
    #
    # Solving this 2x2 linear system gives lambda1 and lambda2.

    V_left_S0   = coeff_left[-1, 0]  + coeff_left[-1, 1]
    Vp_left_S0  = (1.0 / local_vols_left[-1]) * (
                      -coeff_left[-1, 0] + coeff_left[-1, 1])

    V_right_S0  = coeff_right[0, 0]  + coeff_right[0, 1]
    Vp_right_S0 = (1.0 / local_vols_right[0]) * (
                      -coeff_right[0, 0] + coeff_right[0, 1])

    denom   = Vp_left_S0 * V_right_S0 - V_left_S0 * Vp_right_S0
    lambda1 = V_right_S0 / denom
    lambda2 = V_left_S0  / denom

    # Scale all coefficients by their respective lambdas
    coeff_left  = coeff_left  * lambda1
    coeff_right = coeff_right * lambda2

    return (coeff_left, coeff_right,
            float(lambda1), float(lambda2),
            left_nodes, right_nodes,
            local_vols_left, local_vols_right)


# ══════════════════════════════════════════════════════════════════════════════
# SECTION 3: Evaluate Model Call Prices at Market Strikes
#
# Given a theta vector, reconstruct C(K) at each observed market strike.
# This is used both for the bid-ask feasibility check and for plotting.
# ══════════════════════════════════════════════════════════════════════════════

## Section 3: Model Call Prices at Market Strikes

Given $\theta$, we reconstruct $C(K)$ at each observed market strike for the bid-ask feasibility check and plotting. Within interval $j$ with left endpoint $\nu_j^L$:

$$V(K) = \tilde{A}_j \, e^{(d_j - (K - \nu_j^L))/\sigma_j} + \tilde{B}_j \, e^{-(d_j - (K - \nu_j^L))/\sigma_j}$$


In [4]:
def compute_model_calls(theta, market_strikes, R1, R2, S0):
    """
    Compute LVG model call prices C(K) at a set of market strikes.

    C(K) = lambda1 * V_left(K) + (S0 - K)   for K < S0  (left wing)
    C(K) = lambda2 * V_right(K)              for K >= S0 (right wing)

    Parameters
    ----------
    theta          : ndarray (2*R1 + 2*R2,)
    market_strikes : array-like  strike prices to evaluate at
    R1, R2         : int
    S0             : float

    Returns
    -------
    model_calls : ndarray  C(K) at each market strike
    """
    (coeff_left, coeff_right, lambda1, lambda2,
     left_nodes, right_nodes,
     local_vols_left, local_vols_right) = _compute_wing_coefficients(
                                               theta, R1, R2, S0)

    model_calls = []

    for K in market_strikes:
        if K < S0:
            # ── Left wing ──────────────────────────────────────────────
            # Find which interval K falls in
            j = int(np.searchsorted(left_nodes, K, side='right')) - 1
            j = np.clip(j, 0, R1 - 1)

            sig        = local_vols_left[j]
            dist_total = left_nodes[j+1] - left_nodes[j]

            # Recover true coefficients at the LEFT endpoint of interval j
            # (distance-scaled coefficients are relative to the right endpoint)
            cd1 = coeff_left[j, 0] * np.exp( dist_total / sig)
            cd2 = coeff_left[j, 1] * np.exp(-dist_total / sig)

            # Evaluate V(K) as distance from left endpoint of interval
            dist_k = K - left_nodes[j]
            V_K    = cd1 * np.exp(-dist_k / sig) + cd2 * np.exp(dist_k / sig)

            # Call price = time value + intrinsic
            C_K = V_K + (S0 - K)
            model_calls.append(C_K)

        else:
            # ── Right wing ─────────────────────────────────────────────
            j = int(np.searchsorted(right_nodes, K, side='right')) - 1
            j = np.clip(j, 0, R2 - 1)

            sig        = local_vols_right[j]
            dist_total = right_nodes[j+1] - right_nodes[j]

            cd1 = coeff_right[j, 0] * np.exp(-dist_total / sig)
            cd2 = coeff_right[j, 1] * np.exp( dist_total / sig)

            dist_k = K - right_nodes[j]
            V_K    = cd1 * np.exp(-dist_k / sig) + cd2 * np.exp(dist_k / sig)

            # For right wing (OTM calls), intrinsic = 0
            model_calls.append(V_K)

    return np.array(model_calls)


# ══════════════════════════════════════════════════════════════════════════════
# SECTION 4: Objective Function J(theta)
#
# J measures the smoothness of C''(K), which is proportional to the
# risk-neutral density. A smooth C'' means no arbitrage opportunities
# and a well-behaved implied volatility surface.
#
# At each partition point nu_j:
#   C''(nu_j from left)  = (1/sigma_j^2) * V(nu_j)
#   C''(nu_j from right) = (1/sigma_{j+1}^2) * V(nu_j)
#   Jump = C''(right) - C''(left)
#
# J = sum of (Jump)^2 over all partition points (including the S0 junction)
# ══════════════════════════════════════════════════════════════════════════════

## Section 4: Objective Function $J(\theta)$

$C''(K)$ is proportional to the **risk-neutral density** $q(K)$. A smooth $C''$ implies no arbitrage.

At each internal partition point $\nu_{j+1}$ on the left wing:

$$\Delta C''(\nu_{j+1}) = \underbrace{\frac{V(\nu_{j+1})}{\sigma_{j+1}^2}}_{C''(\nu_{j+1}^+)} - \underbrace{\frac{V(\nu_{j+1})}{\sigma_j^2}}_{C''(\nu_{j+1}^-)}$$

Note $V$ is continuous (enforced by $C^1$ matching), so the jump depends only on the change in $\sigma$.

The full objective sums squared jumps over all $R_1 + R_2 - 1$ internal nodes plus the $S_0$ junction:

$$\boxed{J(\theta) = \sum_{j} \left[\Delta C''(\nu_j)\right]^2}$$


In [5]:
def calculate_J(theta, R1, R2, S0):
    """
    Compute J(theta) = sum of squared C'' jumps at all partition points.

    Matches teammates' PyTorch calculate_J exactly.

    Parameters
    ----------
    theta : ndarray (2*R1 + 2*R2,)
    R1, R2 : int
    S0 : float

    Returns
    -------
    J_value : float  the smoothness penalty (lower is better)
    lambda1 : float  left wing scaling constant
    lambda2 : float  right wing scaling constant
    """
    (coeff_left, coeff_right, lambda1, lambda2,
     left_nodes, right_nodes,
     local_vols_left, local_vols_right) = _compute_wing_coefficients(
                                               theta, R1, R2, S0)

    all_jumps = []

    # ── Left wing internal jumps ──────────────────────────────────────────
    # At each partition point left_nodes[j+1] (for j = 0,...,R1-2):
    #   C''(from left)  = (1/sigma_j^2)     * V(left_nodes[j+1])  [end of interval j]
    #   C''(from right) = (1/sigma_{j+1}^2) * V(left_nodes[j+1])  [start of interval j+1]
    #
    # V at end of interval j   = coeff_left[j,0] + coeff_left[j,1]    (distance-scaled)
    # V at start of interval j+1 requires un-scaling by exp(dist/sig)

    for j in range(R1 - 1):
        # C'' at end of interval j (approaching from the left)
        Cpp_from_left = (1.0 / local_vols_left[j]**2) * (
                            coeff_left[j, 0] + coeff_left[j, 1])

        # V at the START of interval j+1 (= left endpoint of interval j+1)
        # coeff_left[j+1] are distance-scaled relative to the right endpoint,
        # so we multiply by exp(dist/sig) to get values at the left endpoint
        dist_j1  = left_nodes[j+2] - left_nodes[j+1]
        sigma_j1 = local_vols_left[j+1]
        V_start_j1 = (coeff_left[j+1, 0] * np.exp( dist_j1 / sigma_j1) +
                      coeff_left[j+1, 1] * np.exp(-dist_j1 / sigma_j1))
        Cpp_from_right = (1.0 / sigma_j1**2) * V_start_j1

        all_jumps.append(Cpp_from_right - Cpp_from_left)

    # ── Junction jump at S0 ───────────────────────────────────────────────
    # The left and right wings meet at S0. Even though V is continuous,
    # C'' can still jump if the local volatility changes across S0.
    Cpp_left_S0  = (1.0 / local_vols_left[-1]**2) * (
                       coeff_left[-1, 0] + coeff_left[-1, 1])
    Cpp_right_S0 = (1.0 / local_vols_right[0]**2) * (
                       coeff_right[0, 0] + coeff_right[0, 1])
    all_jumps.append(Cpp_right_S0 - Cpp_left_S0)

    # ── Right wing internal jumps ─────────────────────────────────────────
    # Same logic as left wing, but propagation direction is reversed.
    # coeff_right[j] are distance-scaled relative to the LEFT endpoint,
    # so evaluating at the RIGHT endpoint = coeff[j,0] + coeff[j,1] directly.

    for j in range(R2 - 1):
        # C'' at start of interval j+1 (approaching from the right)
        Cpp_from_right = (1.0 / local_vols_right[j+1]**2) * (
                             coeff_right[j+1, 0] + coeff_right[j+1, 1])

        # V at the END of interval j (= right endpoint of interval j)
        dist_j  = right_nodes[j+1] - right_nodes[j]
        sigma_j = local_vols_right[j]
        V_end_j = (coeff_right[j, 0] * np.exp(-dist_j / sigma_j) +
                   coeff_right[j, 1] * np.exp( dist_j / sigma_j))
        Cpp_from_left = (1.0 / sigma_j**2) * V_end_j

        all_jumps.append(Cpp_from_right - Cpp_from_left)

    J_value = float(np.sum(np.array(all_jumps)**2))
    return J_value, float(lambda1), float(lambda2)


# ══════════════════════════════════════════════════════════════════════════════
# SECTION 5: Feasibility Check
#
# CMA-ES works in unconstrained space (it can propose any theta).
# Rather than transforming the parameter space to enforce constraints,
# we check feasibility after sampling and assign a large penalty to
# infeasible candidates — they get killed in the selection step.
#
# Three types of constraints:
#   1. Ordering: partition points must be strictly increasing
#   2. Boundary: left points < S0, right points end at Kbar
#   3. Bid-ask: model prices must lie within market bid-ask spread
# ══════════════════════════════════════════════════════════════════════════════

## Section 5: Feasibility Constraints

CMA-ES operates in unconstrained $\mathbb{R}^{2R_1 + 2R_2}$. Infeasible candidates are killed via a large penalty $J_{\text{penalty}} = 10^6$, ensuring they are never selected as parents.

**Three constraint types:**

1. **Ordering:** $\nu_1^L < \nu_2^L < \cdots < \nu_{R_1}^L < S_0$ and $\nu_1^R < \cdots < \nu_{R_2}^R = \bar{K}$
2. **Boundary:** Last right partition point equals $\bar{K} = 2000$ (within tolerance 1.0)
3. **Bid-ask:** $\text{bid}(K_i) \leq C(K_i) \leq \text{ask}(K_i)$ at every market strike $K_i$


In [6]:
INFEASIBLE_PENALTY = 1e6

def is_feasible(theta, R1, R2, S0, Kbar,
                market_strikes=None, bid_prices=None, ask_prices=None):
    """
    Check whether a candidate theta satisfies all model constraints.

    Parameters
    ----------
    theta          : ndarray (2*R1 + 2*R2,)
    R1, R2         : int
    S0, Kbar       : float
    market_strikes : array-like or None   observed market strikes
    bid_prices     : array-like or None   bid prices at each strike
    ask_prices     : array-like or None   ask prices at each strike

    Returns
    -------
    bool  True if all constraints are satisfied, False otherwise
    """
    partition_pts_left  = theta[:R1]
    partition_pts_right = theta[2*R1 : 2*R1 + R2]

    # ── Constraint 1: Left partition points strictly increasing ───────────
    if np.any(np.diff(partition_pts_left) <= 0):
        return False

    # ── Constraint 2: All left partition points must be below S0 ──────────
    if np.any(partition_pts_left >= S0):
        return False

    # ── Constraint 3: Right partition points strictly increasing ──────────
    if np.any(np.diff(partition_pts_right) <= 0):
        return False

    # ── Constraint 4: Last right partition point must equal Kbar ──────────
    # Allow a small tolerance of 1.0 since Kbar=2000 is fixed and we
    # do not want to penalize floating-point rounding errors near the boundary.
    if abs(partition_pts_right[-1] - Kbar) > 1.0:
        return False

    # ── Constraint 5: Bid-ask feasibility (if market data provided) ───────
    # The model call prices C(K) must lie within the observed bid-ask spread
    # at every market strike. This ensures the optimized model is consistent
    # with market prices and no-arbitrage conditions from the 5a step.
    if market_strikes is not None and bid_prices is not None and ask_prices is not None:
        try:
            model_calls = compute_model_calls(theta, market_strikes, R1, R2, S0)

            # Check each strike: bid <= C(K) <= ask
            tol = 1.5
            if np.any(model_calls < bid_prices - tol) or np.any(model_calls > ask_prices + tol):
                return False
        except Exception:
            # If computation fails (e.g., numerical overflow), reject candidate
            return False

    return True


# ══════════════════════════════════════════════════════════════════════════════
# SECTION 6: Candidate Evaluation
#
# Wraps the feasibility check and J computation into a single function.
# This is the function passed to the CMA-ES core as the objective.
# ══════════════════════════════════════════════════════════════════════════════

# Large constant assigned to infeasible or numerically invalid candidates.
# Must be much larger than any realistic J value so infeasible candidates
# are always ranked last and never selected as parents.
INFEASIBLE_PENALTY = 1e6

def evaluate_candidate(theta, R1, R2, S0, Kbar,
                        market_strikes=None, bid_prices=None, ask_prices=None):
    """
    Evaluate J(theta) for one CMA-ES candidate.

    Returns INFEASIBLE_PENALTY if constraints are violated so that
    CMA-ES naturally kills the candidate during selection — it will
    never be in the top mu parents, so it won't influence the mean
    or covariance update.

    Parameters
    ----------
    theta          : ndarray
    R1, R2         : int
    S0, Kbar       : float
    market_strikes : array-like or None
    bid_prices     : array-like or None
    ask_prices     : array-like or None

    Returns
    -------
    float  J value or INFEASIBLE_PENALTY
    """
    if not is_feasible(theta, R1, R2, S0, Kbar,
                       market_strikes, bid_prices, ask_prices):
        return INFEASIBLE_PENALTY

    try:
        J_value, _, _ = calculate_J(theta, R1, R2, S0)
        return J_value if np.isfinite(J_value) else INFEASIBLE_PENALTY
    except Exception:
        return INFEASIBLE_PENALTY


# ══════════════════════════════════════════════════════════════════════════════
# SECTION 7: CMA-ES Core Algorithm
#
# Implements the (mu/w, lambda)-CMA-ES with:
#   - Cumulative Step-size Adaptation (CSA) for sigma
#   - Rank-one + Rank-mu Covariance Matrix Adaptation
#   - Periodic eigendecomposition for efficient sampling
#
# Reference: Hansen, N. (2016). "The CMA Evolution Strategy: A Tutorial."
#            arXiv:1604.00772
#
# The algorithm maintains a multivariate Gaussian search distribution:
#   N(mean, step_size^2 * covariance)
#
# Each generation:
#   1. Sample lambda candidates from the distribution
#   2. Evaluate J for each candidate (infeasible ones get penalty)
#   3. Keep best mu candidates (the "parents")
#   4. Update mean, covariance, step_size using the parents
#   5. Repeat until step_size < tolerance (convergence)
# ══════════════════════════════════════════════════════════════════════════════

## Section 6: CMA-ES Core Algorithm

We implement the $(\mu/w, \lambda)$-CMA-ES with Cumulative Step-size Adaptation (CSA) and rank-one + rank-$\mu$ covariance updates. Reference: Hansen (2016), *arXiv:1604.00772*.

### Search Distribution

Each generation maintains a multivariate Gaussian:

$$\mathcal{N}(m, \sigma^2 C)$$

where $m \in \mathbb{R}^n$ is the mean, $\sigma > 0$ is the global step size, and $C \in \mathbb{R}^{n \times n}$ is the covariance matrix.

### Hyperparameters (Hansen 2016, Table 1)

| Parameter | Formula |
|---|---|
| Population size $\lambda$ | $4 + \lfloor 3 \ln n \rfloor$ |
| Parents $\mu$ | $\lfloor \lambda / 2 \rfloor$ |
| Weights $w_i$ | $\frac{\ln(\mu + \tfrac{1}{2}) - \ln i}{\sum_j [\ln(\mu+\tfrac{1}{2}) - \ln j]}$ |
| Effective mass $\mu_\text{eff}$ | $1 / \sum_i w_i^2$ |

### Per-Generation Update Steps

**1. Sample** $\lambda$ candidates: $x_k = m + \sigma \, B (D \, z_k)$, $z_k \sim \mathcal{N}(0, I)$, where $C = B D^2 B^T$.

**2. Evaluate** $J(x_k)$ for each candidate.

**3. Update mean:** $m \leftarrow \sum_{i=1}^\mu w_i \, x_{i:\lambda}$

**4. Update step-size path $p_\sigma$** (CSA):
$$p_\sigma \leftarrow (1 - c_\sigma) p_\sigma + \sqrt{c_\sigma(2-c_\sigma)\mu_\text{eff}} \, C^{-1/2} \frac{m - m_\text{old}}{\sigma}$$

**5. Update step size $\sigma$:**
$$\sigma \leftarrow \sigma \exp\!\left(\frac{c_\sigma}{d_\sigma}\left(\frac{\|p_\sigma\|}{\mathbb{E}\|N(0,I)\|} - 1\right)\right)$$

**6. Update covariance $C$** (rank-1 + rank-$\mu$):
$$C \leftarrow (1 - c_1 - c_\mu) C + c_1 \, p_c p_c^T + c_\mu \sum_{i=1}^\mu w_i \, y_i y_i^T$$


In [7]:
def _run_cmaes(evaluate_fn, theta_init, initial_step_size=1.0,
               max_generations=500, convergence_tol=1e-10,
               random_seed=42, verbose=True):
    """
    Core CMA-ES optimization loop.

    Parameters
    ----------
    evaluate_fn       : callable theta -> float
                        The objective function (evaluate_candidate)
    theta_init        : ndarray (n,)
                        Starting point for the search distribution mean
    initial_step_size : float
                        Initial value of sigma (global step size).
                        Should be ~10% of the typical parameter gap size.
    max_generations   : int
                        Maximum number of generations before stopping
    convergence_tol   : float
                        Stop when step_size < this value (distribution collapsed)
    random_seed       : int
                        For reproducibility
    verbose           : bool
                        Print progress every 20 generations

    Returns
    -------
    best_theta : ndarray   best parameter vector found across all generations
    best_J     : float     J value at best_theta
    history    : dict      per-generation tracking data
    """
    np.random.seed(random_seed)
    n = len(theta_init)

    # ── CMA-ES Hyperparameters (Hansen 2016, Table 1) ─────────────────────
    #
    # These are the "standard" settings derived from theory. They generally
    # work well without tuning for problems in the 50-300 dimensional range.

    # Population size lambda: roughly 4 + 3*ln(n)
    # More candidates = more exploration but slower per-generation
    population_size = 4 + int(np.floor(3 * np.log(n)))

    # Number of parents mu: best half of the population
    num_parents = population_size // 2

    # Recombination weights: log-spaced so best candidate has highest weight.
    # The 1st candidate gets weight ~log(mu+0.5)-log(1), the mu-th gets ~0.
    raw_weights = np.log(num_parents + 0.5) - np.log(np.arange(1, num_parents + 1))
    weights     = raw_weights / raw_weights.sum()   # normalize to sum=1

    # Variance-effective selection mass: how many "effective" samples are used
    # per generation. mueff ≈ mu/2 for equal weights, mueff ≈ 1 for rank-1.
    mueff = 1.0 / np.sum(weights**2)

    # Step-size control: how fast sigma adapts
    step_size_decay   = (mueff + 2) / (n + mueff + 5)
    step_size_damping = 1 + 2*max(0, np.sqrt((mueff-1)/(n+1))-1) + step_size_decay

    # Expected length of ||N(0,I)|| in n dimensions (for step-size reference)
    expected_norm_sphere = np.sqrt(n) * (1 - 1/(4*n) + 1/(21*n**2))

    # Covariance matrix adaptation rates
    cov_path_decay = (4 + mueff/n) / (n + 4 + 2*mueff/n)
    cov_rank_one   = 2 / ((n + 1.3)**2 + mueff)          # rank-1 learning rate
    cov_rank_mu    = min(1 - cov_rank_one,
                        2*(mueff-2+1/mueff) / ((n+2)**2 + mueff))  # rank-mu rate

    # Threshold for the h_sigma indicator (suppresses rank-1 update if needed)
    h_sigma_thresh = (1.4 + 2/(n+1)) * expected_norm_sphere

    if verbose:
        print(f"\n{'='*65}")
        print(f"  CMA-ES Optimizer  (v4 — direct space + bid-ask constraints)")
        print(f"{'='*65}")
        print(f"  Dimension n          = {n}")
        print(f"  Population size λ    = {population_size}")
        print(f"  Number of parents μ  = {num_parents}")
        print(f"  Effective mass μeff  = {mueff:.2f}")
        print(f"  Initial step size σ₀ = {initial_step_size}")
        print(f"  Max generations      = {max_generations}")
        print(f"  Convergence tol      = {convergence_tol:.1e}")
        print(f"{'='*65}")
        print(f"  {'Gen':>5} | {'Best J':>14} | {'Feasible':>12} | {'σ':>10}")
        print(f"  {'-'*50}")

    # ── State Variables ───────────────────────────────────────────────────

    mean              = theta_init.copy()   # current center of search distribution
    step_size         = initial_step_size   # global step size sigma

    # Evolution paths accumulate the history of mean movements
    evolution_path_C  = np.zeros(n)   # used for rank-1 covariance update
    evolution_path_s  = np.zeros(n)   # used for step-size adaptation

    # Covariance matrix C and its eigendecomposition
    # C = eigenvectors @ diag(axis_lengths^2) @ eigenvectors.T
    eigenvectors      = np.eye(n)     # B: columns are eigenvectors of C
    axis_lengths      = np.ones(n)    # D: sqrt of eigenvalues (axis lengths)
    covariance        = np.eye(n)     # C: the full covariance matrix
    inv_sqrt_cov      = np.eye(n)     # C^{-1/2}: for step-size path normalization

    last_eigen_update = 0             # track when we last decomposed C

    best_J            = np.inf
    best_theta        = mean.copy()

    # History for plotting and diagnostics
    history = {
        'best_J_per_gen':  [],   # best J found so far at each generation
        'mean_J_per_gen':  [],   # mean J across the population (includes penalties)
        'step_size':       [],   # sigma at each generation
        'feasible_count':  [],   # number of feasible candidates per generation
        'generation':      []
    }

    # ── Generation Loop ───────────────────────────────────────────────────
    for generation in range(max_generations):

        # ── Step 1: Sample lambda candidates from N(mean, sigma^2 * C) ───
        #
        # Efficient sampling using eigendecomposition:
        #   x_k = mean + sigma * B * (D * z_k),  z_k ~ N(0, I)
        #
        # This is equivalent to sampling from N(mean, sigma^2 * C) because:
        #   Cov[B*(D*z)] = B * diag(D^2) * B^T = C
        #
        # B (eigenvectors) rotates the standard normal into the ellipse axes
        # D (axis_lengths)  stretches each axis by the corresponding eigenvalue

        standard_samples = np.random.randn(population_size, n)
        candidates = np.array([
            mean + step_size * (eigenvectors @ (axis_lengths * standard_samples[k]))
            for k in range(population_size)
        ])

        # ── Step 2: Evaluate objective for each candidate ─────────────────
        # Infeasible candidates receive INFEASIBLE_PENALTY (1e6) and will
        # be ranked last — they never influence the parameter updates.
        fitness_values = np.array([
            evaluate_fn(candidates[k]) for k in range(population_size)
        ])

        # Count how many candidates were feasible this generation
        num_feasible = int(np.sum(fitness_values < INFEASIBLE_PENALTY))

        # ── Step 3: Sort by fitness (ascending = minimization) ────────────
        sorted_indices = np.argsort(fitness_values)

        # ── Step 4: Update best solution found so far ─────────────────────
        if fitness_values[sorted_indices[0]] < best_J:
            best_J     = fitness_values[sorted_indices[0]]
            best_theta = candidates[sorted_indices[0]].copy()

        # ── Step 5: Update mean (weighted average of best mu parents) ─────
        #
        # New mean = weighted sum of the best num_parents candidates.
        # Better candidates (lower J) get higher weights.
        # This is the "gradient-free gradient step" — we move toward
        # where the best samples were found.
        old_mean = mean.copy()
        mean = np.sum(
            weights[:, None] * candidates[sorted_indices[:num_parents]],
            axis=0
        )

        # ── Step 6: Update step-size evolution path p_s ───────────────────
        #
        # p_s accumulates normalized mean movements.
        # If steps are consistently in the same direction, ||p_s|| grows.
        # If steps oscillate back and forth, ||p_s|| stays small.
        # C^{-1/2} normalizes the step so its expected length is sqrt(n).
        evolution_path_s = (
            (1 - step_size_decay) * evolution_path_s
            + np.sqrt(step_size_decay * (2 - step_size_decay) * mueff)
            * inv_sqrt_cov @ (mean - old_mean) / step_size
        )

        # ── Step 7: Compute h_sigma (Heaviside indicator) ─────────────────
        #
        # h_sigma = 1 if the evolution path is "short enough" to be reliable.
        # h_sigma = 0 suppresses the rank-1 covariance update in the early
        # generations when the path hasn't accumulated enough history.
        path_length_norm = (
            np.linalg.norm(evolution_path_s)
            / np.sqrt(1 - (1 - step_size_decay)**(2*(generation+1)))
        )
        h_sigma = (path_length_norm < h_sigma_thresh)

        # ── Step 8: Update covariance evolution path p_c ──────────────────
        #
        # p_c accumulates the history of mean movements (without C^{-1/2}).
        # It is used for the rank-1 covariance update below.
        # The outer product p_c * p_c^T encodes the direction of consistent
        # improvement across generations.
        evolution_path_C = (
            (1 - cov_path_decay) * evolution_path_C
            + h_sigma * np.sqrt(cov_path_decay * (2 - cov_path_decay) * mueff)
            * (mean - old_mean) / step_size
        )

        # ── Step 9: Update covariance matrix C ────────────────────────────
        #
        # C is updated by two mechanisms:
        #
        # Rank-1 update (cov_rank_one):
        #   Uses the single evolution path p_c. This captures the direction
        #   of consistent improvement accumulated across many generations.
        #   Weight = cov_rank_one * p_c * p_c^T
        #
        # Rank-mu update (cov_rank_mu):
        #   Uses all num_parents successful steps from this generation.
        #   The weighted sum of outer products captures the local curvature.
        #   Weight = cov_rank_mu * sum_i w_i * y_i * y_i^T
        #
        # The old covariance decays at rate (1 - cov_rank_one - cov_rank_mu).

        # Normalized steps of the best parents (relative to old mean, scaled by sigma)
        normalized_steps = (
            (1.0 / step_size)
            * (candidates[sorted_indices[:num_parents]] - old_mean)
        )

        covariance = (
            (1 - cov_rank_one - cov_rank_mu) * covariance
            # Rank-1 update: from evolution path history
            + cov_rank_one * (
                np.outer(evolution_path_C, evolution_path_C)
                + (1 - h_sigma) * cov_path_decay * (2 - cov_path_decay) * covariance
            )
            # Rank-mu update: from current generation's best candidates
            + cov_rank_mu * np.sum(
                weights[:, None, None]
                * (normalized_steps[:, :, None] * normalized_steps[:, None, :]),
                axis=0
            )
        )

        # ── Step 10: Update step size sigma (CSA) ─────────────────────────
        #
        # sigma grows when ||p_s|| > expected_norm_sphere (correlated steps = good)
        # sigma shrinks when ||p_s|| < expected_norm_sphere (oscillating steps = near min)
        step_size = step_size * np.exp(
            (step_size_decay / step_size_damping)
            * (np.linalg.norm(evolution_path_s) / expected_norm_sphere - 1)
        )

        # ── Step 11: Periodic eigendecomposition of C ─────────────────────
        #
        # Computing B and D from C is O(n^3) — expensive for n=209.
        # We only do it every ~n/(10*lambda) generations.
        # Between updates, we use the stored B and D for fast sampling.
        eigendecomp_interval = n / (10 * population_size * (cov_rank_one + cov_rank_mu))
        if generation - last_eigen_update > eigendecomp_interval:
            last_eigen_update = generation

            # Enforce symmetry (numerical drift can make C slightly asymmetric)
            covariance = np.triu(covariance) + np.triu(covariance, 1).T

            # Eigendecomposition: C = B * diag(D^2) * B^T
            eigenvalues_sq, eigenvectors = np.linalg.eigh(covariance)

            # Floor eigenvalues at 1e-20 to prevent numerical issues
            eigenvalues_sq = np.maximum(eigenvalues_sq, 1e-20)
            axis_lengths   = np.sqrt(eigenvalues_sq)        # D = sqrt(eigenvalues)

            # C^{-1/2} = B * diag(1/D) * B^T (for step-size path normalization)
            inv_sqrt_cov = eigenvectors @ np.diag(1.0 / axis_lengths) @ eigenvectors.T

        # ── Step 12: Record history for diagnostics and plotting ───────────
        history['best_J_per_gen'].append(best_J)
        history['mean_J_per_gen'].append(float(np.mean(fitness_values)))
        history['step_size'].append(float(step_size))
        history['feasible_count'].append(num_feasible)
        history['generation'].append(generation)

        # ── Step 13: Print progress ────────────────────────────────────────
        if verbose and generation % 20 == 0:
            print(f"  {generation:>5} | {best_J:>14.8f} | "
                  f"{num_feasible:>5}/{population_size:<5} feasible | "
                  f"σ={step_size:>10.6f}")

        # ── Step 14: Check convergence ─────────────────────────────────────
        # Primary stopping criterion: step size has collapsed
        # This means the search distribution has shrunk to a point —
        # CMA-ES has converged and further generations won't help.
        if step_size < convergence_tol:
            if verbose:
                print(f"\n  Converged at gen {generation}: "
                      f"σ = {step_size:.2e} < tol = {convergence_tol:.2e}")
            break

        # Secondary: J is essentially zero (perfect smoothness)
        if best_J < 1e-12:
            if verbose:
                print(f"\n  Converged at gen {generation}: J = {best_J:.2e} ≈ 0")
            break

    if verbose:
        print(f"  {'-'*50}")

    return best_theta, best_J, history


# ══════════════════════════════════════════════════════════════════════════════
# SECTION 8: Public Optimization API
# ══════════════════════════════════════════════════════════════════════════════

## Section 7: Public Optimization API

`cmaes_optimize` wraps the core loop with logging, plotting, and baseline comparison.


In [8]:
def cmaes_optimize(theta_init, R1, R2, S0, Kbar,
                   theta_baseline=None,
                   market_strikes=None,
                   bid_prices=None,
                   ask_prices=None,
                   initial_step_size=1.0,
                   max_generations=500,
                   convergence_tol=1e-10,
                   random_seed=42,
                   verbose=True,
                   save_plot=True,
                   plot_path='cmaes_result.png'):
    """
    Minimize J(theta) using CMA-ES in direct parameter space.

    Infeasible candidates (ordering violations, boundary violations,
    or bid-ask violations) are killed by the penalty function rather
    than excluded by a coordinate transform.

    Parameters
    ----------
    theta_init        : ndarray (2*R1 + 2*R2,)
                        Starting theta for THIS optimization call.
                        In a sigma schedule, this changes each stage.

    R1, R2            : int   number of left/right partition points
    S0                : float spot price (S0 = 1271.87 for SP500 data)
    Kbar              : float right boundary (fixed = 2000.0)

    theta_baseline    : ndarray or None
                        The ORIGINAL pre-optimization LVG theta.
                        Used as the "before" curve in all plots.
                        If None, uses theta_init (backward compatible).

    market_strikes    : array-like or None
                        Observed market strikes from Quotes.csv.
                        If provided, bid-ask constraint is enforced.
    bid_prices        : array-like or None   bid prices at each strike
    ask_prices        : array-like or None   ask prices at each strike

    initial_step_size : float   CMA-ES initial sigma (default 1.0)
                        Tip: use ~10% of typical partition gap (~0.3-0.5)
    max_generations   : int     maximum generations (default 500)
    convergence_tol   : float   stop when sigma < this (default 1e-10)
    random_seed       : int     for reproducibility (default 42)
    verbose           : bool    print per-generation progress (default True)
    save_plot         : bool    save comparison plot (default True)
    plot_path         : str     file path for the output plot

    Returns
    -------
    dict with keys:
        'best_theta'   ndarray  optimized parameter vector
        'best_J'       float    optimized J value at best_theta
        'J_initial'    float    J value at theta_init (start of this call)
        'J_baseline'   float    J value at theta_baseline (original LVG)
        'improvement'  float    % improvement vs theta_baseline
        'lambda1_opt'  float    lambda1 at best_theta
        'lambda2_opt'  float    lambda2 at best_theta
        'history'      dict     per-generation convergence data
    """

    # Use theta_init as baseline if no separate baseline provided
    if theta_baseline is None:
        theta_baseline = theta_init

    # ── Compute J values ──────────────────────────────────────────────────
    J_baseline, _, _           = calculate_J(theta_baseline, R1, R2, S0)
    J_initial, lambda1_init, lambda2_init = calculate_J(theta_init, R1, R2, S0)

    if verbose:
        print(f"\n  Original LVG baseline J  = {J_baseline:.8f}")
        print(f"  Starting J (this stage)  = {J_initial:.8f}")
        print(f"  Lambda1 (init)           = {lambda1_init:.6f}")
        print(f"  Lambda2 (init)           = {lambda2_init:.6f}")
        if market_strikes is not None:
            print(f"  Bid-ask constraint       = ACTIVE ({len(market_strikes)} strikes)")
        else:
            print(f"  Bid-ask constraint       = OFF (no market data provided)")

    # ── Wrap evaluate_candidate with fixed parameters ─────────────────────
    # Convert market data to numpy arrays once, outside the inner loop
    ms  = np.array(market_strikes)  if market_strikes is not None else None
    bid = np.array(bid_prices)      if bid_prices     is not None else None
    ask = np.array(ask_prices)      if ask_prices     is not None else None

    def objective(theta):
        return evaluate_candidate(theta, R1, R2, S0, Kbar, ms, bid, ask)

    # ── Run CMA-ES ────────────────────────────────────────────────────────
    best_theta, best_J, history = _run_cmaes(
        evaluate_fn       = objective,
        theta_init        = theta_init,
        initial_step_size = initial_step_size,
        max_generations   = max_generations,
        convergence_tol   = convergence_tol,
        random_seed       = random_seed,
        verbose           = verbose
    )

    # ── Compute final lambdas ─────────────────────────────────────────────
    _, lambda1_opt, lambda2_opt = calculate_J(best_theta, R1, R2, S0)

    # Improvement is always measured vs the ORIGINAL LVG baseline
    improvement = 100.0 * (J_baseline - best_J) / J_baseline if J_baseline > 0 else 0.0

    if verbose:
        print(f"\n{'='*65}")
        print(f"  Stage Complete")
        print(f"{'='*65}")
        print(f"  Original baseline J  = {J_baseline:.8f}")
        print(f"  This stage start J   = {J_initial:.8f}")
        print(f"  This stage best J    = {best_J:.8f}")
        print(f"  Improvement vs LVG   = {improvement:.4f}%")
        print(f"  Generations run      = {len(history['generation'])}")
        print(f"  Lambda1: {lambda1_init:.4f} -> {lambda1_opt:.4f}")
        print(f"  Lambda2: {lambda2_init:.4f} -> {lambda2_opt:.4f}")
        print(f"{'='*65}\n")

    # ── Save comparison plot ──────────────────────────────────────────────
    if save_plot:
        _generate_plots(
            history        = history,
            J_baseline     = J_baseline,
            best_J         = best_J,
            improvement    = improvement,
            theta_baseline = theta_baseline,    # always the original LVG output
            best_theta     = best_theta,
            R1=R1, R2=R2, S0=S0,
            market_strikes = ms,
            bid_prices     = bid,
            ask_prices     = ask,
            save_path      = plot_path
        )
        if verbose:
            print(f"  Plot saved to: {plot_path}")
    return {
        'best_theta':  best_theta,
        'best_J':      best_J,
        'J_initial':   J_initial,
        'J_baseline':  J_baseline,
        'improvement': improvement,
        'lambda1_opt': lambda1_opt,
        'lambda2_opt': lambda2_opt,
        'history':     history
    }

# ══════════════════════════════════════════════════════════════════════════════
# SECTION 9: Plotting Utilities
# ══════════════════════════════════════════════════════════════════════════════

## Section 8: Plotting Utilities

Three diagnostic functions:
- `_get_jumps`: compute all $\Delta C''(\nu_j)$ values from a theta vector
- `_get_curves`: evaluate $C(K)$ and $C''(K)$ on a fine grid within each interval
- `_generate_plots`: three-panel figure (convergence, call prices, second derivative)


In [9]:
def _get_jumps(theta, R1, R2, S0):
    """
    Compute all C'' jump values from a theta vector.
    Returns an array of length (R1-1) + 1 + (R2-1) = R1+R2-1 jumps.
    """
    (coeff_left, coeff_right, lambda1, lambda2,
     left_nodes, right_nodes,
     local_vols_left, local_vols_right) = _compute_wing_coefficients(
                                               theta, R1, R2, S0)
    jumps = []

    # Left wing internal jumps
    for j in range(R1-1):
        dist = left_nodes[j+2]-left_nodes[j+1]; s = local_vols_left[j+1]
        vr   = (1/s**2)*(coeff_left[j+1,0]*np.exp(dist/s)
                         +coeff_left[j+1,1]*np.exp(-dist/s))
        vl   = (1/local_vols_left[j]**2)*(coeff_left[j,0]+coeff_left[j,1])
        jumps.append(vr-vl)

    # Junction jump at S0
    jumps.append(
        (1/local_vols_right[0]**2)*(coeff_right[0,0]+coeff_right[0,1])
        - (1/local_vols_left[-1]**2)*(coeff_left[-1,0]+coeff_left[-1,1])
    )

    # Right wing internal jumps
    for j in range(R2-1):
        dist = right_nodes[j+1]-right_nodes[j]; s = local_vols_right[j]
        vl   = (1/s**2)*(coeff_right[j,0]*np.exp(-dist/s)
                         +coeff_right[j,1]*np.exp(dist/s))
        vr   = (1/local_vols_right[j+1]**2)*(coeff_right[j+1,0]
                                               +coeff_right[j+1,1])
        jumps.append(vr-vl)

    return np.array(jumps)

def _get_curves(theta, R1, R2, S0, n_pts=50):
    """
    Compute C(K) and C''(K) on a fine grid within each interval.

    Returns
    -------
    K_arr   : ndarray  sorted strike values
    C_arr   : ndarray  call prices C(K)
    Cpp_arr : ndarray  second derivative C''(K)
    """
    (coeff_left, coeff_right, lambda1, lambda2,
     left_nodes, right_nodes,
     local_vols_left, local_vols_right) = _compute_wing_coefficients(
                                               theta, R1, R2, S0)

    K_all = []; C_all = []; Cpp_all = []

    # ── Left wing intervals ───────────────────────────────────────────────
    for j in range(R1):
        nu_L = left_nodes[j]; nu_R = left_nodes[j+1]; sig = local_vols_left[j]
        dist_total = nu_R - nu_L

        # Recover true coefficients at LEFT endpoint of interval j
        cd1 = coeff_left[j, 0] * np.exp( dist_total / sig)
        cd2 = coeff_left[j, 1] * np.exp(-dist_total / sig)

        for k in np.linspace(nu_L, nu_R, n_pts):
            dist_k = k - nu_L
            V_k    = cd1 * np.exp(-dist_k / sig) + cd2 * np.exp(dist_k / sig)
            K_all.append(k)
            C_all.append(V_k + max(S0 - k, 0))   # C(K) = V(K) + intrinsic
            Cpp_all.append(V_k / sig**2)           # C''(K) = V(K) / sigma^2

    # ── Right wing intervals ──────────────────────────────────────────────
    for j in range(R2):
        nu_L = right_nodes[j]; nu_R = right_nodes[j+1]; sig = local_vols_right[j]
        dist_total = nu_R - nu_L

        cd1 = coeff_right[j, 0] * np.exp(-dist_total / sig)
        cd2 = coeff_right[j, 1] * np.exp( dist_total / sig)

        for k in np.linspace(nu_L, nu_R, n_pts):
            dist_k = k - nu_L
            V_k    = cd1 * np.exp(-dist_k / sig) + cd2 * np.exp(dist_k / sig)
            K_all.append(k)
            C_all.append(V_k)                      # C(K) = V(K), no intrinsic
            Cpp_all.append(V_k / sig**2)

    K_arr   = np.array(K_all)
    C_arr   = np.array(C_all)
    Cpp_arr = np.array(Cpp_all)
    idx     = np.argsort(K_arr)
    return K_arr[idx], C_arr[idx], Cpp_arr[idx]

def _generate_plots(history, J_baseline, best_J, improvement,
                    theta_baseline, best_theta, R1, R2, S0,
                    market_strikes=None, bid_prices=None, ask_prices=None,
                    save_path='cmaes_result.png'):
    """
    Generate and save a three-panel comparison figure.

    Always compares BEST_THETA (optimized) against THETA_BASELINE
    (the original pre-optimization LVG output), regardless of which
    stage of the optimization we are in.

    Panel 1: J convergence curve (log scale)
    Panel 2: C(K) call price — baseline vs optimized, with bid/ask overlay
    Panel 3: C''(K) second derivative — zoomed to active strike range
    """
    generations    = history['generation']

    # Compute curves for baseline (original LVG) and optimized theta
    K_base,  C_base,  Cpp_base  = _get_curves(theta_baseline, R1, R2, S0)
    K_opt,   C_opt,   Cpp_opt   = _get_curves(best_theta,     R1, R2, S0)

    jumps_base = _get_jumps(theta_baseline, R1, R2, S0)
    jumps_opt  = _get_jumps(best_theta,     R1, R2, S0)

    # ── Determine zoom range for C and C'' plots ──────────────────────────
    # Focus on the region where market strikes are observed (the liquid range).
    # This is more informative than showing the full [0, 2000] range.
    if market_strikes is not None and len(market_strikes) > 0:
        K_lo = float(np.min(market_strikes)) - 30
        K_hi = float(np.max(market_strikes)) + 30
    else:
        # Default zoom: 150 units either side of S0
        K_lo = S0 - 150
        K_hi = S0 + 200

    fig, axes = plt.subplots(3, 1, figsize=(13, 16))
    fig.suptitle(
        f'CMA-ES Optimization Results\n'
        f'J: {J_baseline:.6f} (LVG baseline)  →  {best_J:.2e} (optimized)'
        f'   [{improvement:.2f}% improvement]',
        fontsize=13, fontweight='bold', y=1.01
    )

    # ── Panel 1: J Convergence Curve ─────────────────────────────────────
    ax = axes[0]
    ax.semilogy(generations, history['best_J_per_gen'],
                'b-', lw=2, label='Best J found so far')
    ax.axhline(J_baseline, color='tomato', ls=':', lw=2.5,
               label=f'LVG baseline J = {J_baseline:.6f}')
    ax.axhline(best_J, color='green', ls='--', lw=1.5,
               label=f'Optimized J* = {best_J:.2e}')
    ax.set_xlabel('Generation', fontsize=12)
    ax.set_ylabel('J(θ)  [log scale]', fontsize=12)
    ax.set_title('J(θ) Convergence  —  lower is better', fontsize=12)
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)

    # ── Panel 2: C(K) Call Price ──────────────────────────────────────────
    # Show model call prices for both baseline and optimized theta.
    # Overlay bid/ask market prices as scatter points to verify
    # the model stays within the feasible region.
    ax = axes[1]

    # Filter to zoom range
    mask_b = (K_base >= K_lo) & (K_base <= K_hi)
    mask_o = (K_opt  >= K_lo) & (K_opt  <= K_hi)

    ax.plot(K_base[mask_b], C_base[mask_b],
            'r-', lw=1.8, alpha=0.8, label='C(K) — LVG baseline')
    ax.plot(K_opt[mask_o],  C_opt[mask_o],
            'b-', lw=1.8, alpha=0.8, label='C(K) — CMA-ES optimized')

    # Overlay market bid/ask if available
    if market_strikes is not None and bid_prices is not None and ask_prices is not None:
        ms  = np.array(market_strikes)
        bid = np.array(bid_prices)
        ask = np.array(ask_prices)
        mask_mkt = (ms >= K_lo) & (ms <= K_hi)
        ax.scatter(ms[mask_mkt], ask[mask_mkt],
                   color='gray', s=25, marker='^', zorder=5, label='Ask')
        ax.scatter(ms[mask_mkt], bid[mask_mkt],
                   color='gray', s=25, marker='v', zorder=5, label='Bid')

    ax.axvline(S0, color='gray', ls='--', lw=1.5, label=f'S₀ = {S0:.0f}')
    ax.set_xlim(K_lo, K_hi)
    ax.set_xlabel('Strike K', fontsize=12)
    ax.set_ylabel('C(K)', fontsize=12)
    ax.set_title('Call Price C(K)  —  Baseline vs Optimized', fontsize=12)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

    # ── Panel 3: C''(K) Second Derivative (Zoomed) ───────────────────────
    # C''(K) is proportional to the risk-neutral density.
    # A smooth C'' means no arbitrage. The jumps at partition points
    # are exactly what J measures — we want them to be small.
    ax = axes[2]

    # Clip outliers for clean display (a few extreme values at interval edges)
    Cpp_combined = np.concatenate([Cpp_base, Cpp_opt])
    clip_lo = np.percentile(Cpp_combined, 2)
    clip_hi = np.percentile(Cpp_combined, 98)

    mask_b = (K_base >= K_lo) & (K_base <= K_hi)
    mask_o = (K_opt  >= K_lo) & (K_opt  <= K_hi)

    ax.plot(K_base[mask_b], np.clip(Cpp_base[mask_b], clip_lo, clip_hi),
            'r-', lw=1.5, alpha=0.85, label="C''(K) — LVG baseline")
    ax.plot(K_opt[mask_o],  np.clip(Cpp_opt[mask_o],  clip_lo, clip_hi),
            'b-', lw=1.5, alpha=0.85, label="C''(K) — CMA-ES optimized")

    ax.axvline(S0, color='gray', ls='--', lw=1.5, label=f'S₀ = {S0:.0f}')
    ax.axhline(0,  color='black', lw=0.5)
    ax.set_xlim(K_lo, K_hi)
    ax.set_xlabel('Strike K', fontsize=12)
    ax.set_ylabel("C''(K)", fontsize=12)
    ax.set_title(
        f"C''(K) Second Derivative  —  Baseline vs Optimized  "
        f"(Strike range: {K_lo:.0f}–{K_hi:.0f})\n"
        f"Max |jump|: {np.max(np.abs(jumps_base)):.5f} → "
        f"{np.max(np.abs(jumps_opt)):.5f}   |   "
        f"Mean |jump|: {np.mean(np.abs(jumps_base)):.5f} → "
        f"{np.mean(np.abs(jumps_opt)):.5f}",
        fontsize=11
    )
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()

    # ── Print summary to console ──────────────────────────────────────────
    print(f"\n  ── Plot Summary ──────────────────────────────────────")
    print(f"    J (LVG baseline)  = {J_baseline:.8f}")
    print(f"    J (optimized)     = {best_J:.8f}")
    print(f"    Improvement       = {improvement:.4f}%")
    print(f"    Max |ΔC''|        : {np.max(np.abs(jumps_base)):.6f}"
          f"  →  {np.max(np.abs(jumps_opt)):.6f}"
          f"  ({100*(np.max(np.abs(jumps_base))-np.max(np.abs(jumps_opt)))/np.max(np.abs(jumps_base)):.1f}% reduction)")
    print(f"    Mean|ΔC''|        : {np.mean(np.abs(jumps_base)):.6f}"
          f"  →  {np.mean(np.abs(jumps_opt)):.6f}"
          f"  ({100*(np.mean(np.abs(jumps_base))-np.mean(np.abs(jumps_opt)))/np.mean(np.abs(jumps_base)):.1f}% reduction)")
    print(f"  ─────────────────────────────────────────────────────")

## Section 9: Main Execution

Three-step optimization strategy:

1. **Sigma schedule** — start with $\sigma_0 = 0.3$ (broad exploration), progressively reduce to $\sigma_0 = 0.0005$ (fine refinement). Each stage warm-starts from the previous best.
2. **Multi-start refinement** — 20 trials from slightly perturbed starting points ($\epsilon = 10^{-4}$), keeping the global best.

| Stage | $\sigma_0$ | Max generations |
|---|---|---|
| 1 | 0.3 | 300 |
| 2 | 0.1 | 300 |
| 3 | 0.05 | 300 |
| 4 | 0.01 | 500 |
| 5 | 0.005 | 500 |
| 6 | 0.001 | 1000 |
| 7 | 0.0005 | 1000 |


In [10]:
import numpy as np

# ── Model dimensions and parameters ──────────────────────────────────
R1   = 70        # number of left-wing partition points
R2   = 34        # number of right-wing partition points (includes Kbar)
S0   = 1271.87   # current underlying price (SP500)
Kbar = 2000.0    # right boundary (far OTM, essentially 0 call price)

# ── Market data ───────────────────────────────────────────────────────
# Load bid-ask prices from Quotes.csv (Strike, Ask, Bid)
# These define the feasible region: model prices must lie between bid and ask.
# Change the path below to wherever Quotes.csv is on your machine.
QUOTES_PATH = '/Users/lucasxia/Downloads/Quotes.csv'  # <-- update this path if needed

market_strikes, ask_prices, bid_prices = [], [], []
try:
    with open(QUOTES_PATH) as f:
        for row in csv.reader(f):
            market_strikes.append(float(row[0]))
            ask_prices.append(float(row[1]))
            bid_prices.append(float(row[2]))
    market_strikes = np.array(market_strikes)
    ask_prices     = np.array(ask_prices)
    bid_prices     = np.array(bid_prices)
    print(f"Loaded {len(market_strikes)} market strikes from {QUOTES_PATH}")
    print(f"Strike range: {market_strikes.min():.0f} to {market_strikes.max():.0f}")
except FileNotFoundError:
    print(f"Warning: {QUOTES_PATH} not found — bid-ask constraint will be OFF.")
    print("Set QUOTES_PATH to the correct location of Quotes.csv.")
    market_strikes = ask_prices = bid_prices = None

# ── Initial LVG calibration output ───────────────────────────────────
# These values come from running the 5a (analytic center) + 5b (LVG interpolation)
# pipeline on the SP500 options data. They are the starting point for optimization.

# Left-wing local volatilities (sigma_j for j=1..R1)
initial_sigs1 = np.array([
    610.43, 164.05, 21.41, 41.04, 32.59, 48.78, 42.22, 55.82, 51.21, 60.69,
    59.95, 58.90, 61.25, 58.78, 61.94, 57.79, 62.28, 54.62, 60.39, 50.96,
    57.50, 46.58, 54.08, 40.60, 46.80, 40.23, 44.18, 40.07, 42.59, 41.97,
    43.55, 43.64, 46.68, 40.67, 45.75, 37.59, 43.37, 34.28, 38.64, 34.12,
    38.34, 33.12, 37.54, 30.90, 34.63, 31.93, 34.99, 31.44, 34.80, 31.01,
    33.67, 32.96, 33.88, 37.49, 36.95, 42.71, 46.47, 33.85, 43.68, 24.47,
    29.27, 33.25, 32.86, 36.88, 38.57, 34.89, 32.34, 54.86, 42.54, 31.82
])

# Right-wing local volatilities (sigma_j for j=1..R2)
initial_sigs2 = np.array([
    24.03, 38.31, 64.22, 24.95, 31.48, 31.17, 29.04, 30.23, 28.25, 27.35,
    26.08, 24.72, 23.36, 22.45, 22.00, 19.57, 19.32, 17.76, 19.39, 39.62,
    26.39, 49.90, 28.53, 31.87, 41.72, 83.05, 40.34, 37.01, 39.35, 30.27,
    73.53, 57.33, 169.44, 313.04
])

# Left-wing partition points (nu_j for j=1..R1, must be < S0)
initial_nus1 = np.array([
    0, 985.90, 1103.9, 1105.63, 1108.9, 1110.93, 1113.9, 1116.08, 1118.9,
    1121.22, 1123.9, 1126.45, 1128.9, 1131.48, 1133.9, 1136.52, 1138.9,
    1141.60, 1143.9, 1146.64, 1148.9, 1151.70, 1153.9, 1156.79, 1158.9,
    1161.63, 1163.9, 1166.56, 1168.9, 1171.46, 1173.9, 1176.44, 1178.9,
    1181.62, 1183.9, 1186.69, 1188.9, 1191.74, 1193.9, 1196.55, 1198.8,
    1201.53, 1203.8, 1206.59, 1208.8, 1211.46, 1213.8, 1216.49, 1218.8,
    1221.50, 1223.8, 1226.39, 1228.8, 1231.23, 1233.8, 1236.18, 1238.8,
    1241.75, 1243.8, 1247.06, 1248.8, 1251.20, 1253.8, 1256.22, 1258.8,
    1261.48, 1263.8, 1265.71, 1268.8, 1270.58
])

# Right-wing partition points (nu_j for j=1..R2, last value = Kbar = 2000.0)
initial_nus2 = np.array([
    1272.60, 1273.8, 1277.34, 1278.8, 1281.24, 1283.8, 1286.17, 1288.8,
    1291.26, 1293.8, 1296.28, 1298.8, 1301.21, 1303.7, 1306.26, 1308.7,
    1311.23, 1313.7, 1316.81, 1323.7, 1327.01, 1333.7, 1336.02, 1338.7,
    1343.47, 1353.7, 1356.28, 1358.7, 1361.50, 1363.7, 1377.34, 1388.7,
    1503.72, 2000.0
])

# ── Pack theta ────────────────────────────────────────────────────────
# The softplus transform means we store raw sigma values in theta,
# not the actual sigma values. inv_softplus converts back.
# This ensures softplus(sigs_raw) + 1e-6 = actual sigma everywhere.
sigs1_raw = inv_softplus(initial_sigs1)
sigs2_raw = inv_softplus(initial_sigs2)

theta_init = np.concatenate([initial_nus1, sigs1_raw,
                              initial_nus2, sigs2_raw])
print(f"\nParameter vector length = {len(theta_init)} "
      f"(expected 2*R1 + 2*R2 = {2*R1+2*R2})")

# ── Compute and display baseline J ────────────────────────────────────
J_baseline, lam1_base, lam2_base = calculate_J(theta_init, R1, R2, S0)
print(f"\nOriginal LVG baseline J = {J_baseline:.8f}")
print(f"Baseline lambda1 = {lam1_base:.6f}")
print(f"Baseline lambda2 = {lam2_base:.6f}")
print("=" * 60)

# ── Step 1 + 2: Sigma Schedule with Generation Schedule ───────────────
#
# Strategy: start with large sigma to explore broadly, then progressively
# reduce sigma to refine within the best basin found so far.
#
# Each stage starts from the best theta found in the previous stage,
# so we never lose progress. The key insight is:
#   - Large sigma: CMA-ES samples widely, finds a good basin quickly.
#     Many candidates may violate ordering constraints.
#   - Small sigma: CMA-ES makes precise adjustments. More candidates
#     are feasible (small steps rarely jump over a partition point).
#
# Sigma values are chosen to be ~10x the typical partition gap at coarse
# stages, shrinking to ~0.01x the gap at the finest stage.
# Typical partition gap ≈ 4 units, so sigma schedule starts at 0.3.
#
# Generation counts increase for fine stages because CMA-ES needs more
# time to adapt its covariance matrix at small step sizes.

sigma_schedule       = [0.3,   0.1,   0.05,  0.01,  0.005, 0.001, 0.0005]
generations_schedule = [300,   300,   300,   500,   500,   1000,  1000  ]

current_theta = theta_init.copy()   # start from the original LVG output

for stage, (sigma, n_gen) in enumerate(zip(sigma_schedule, generations_schedule)):
    print(f"\nStage {stage+1}/{len(sigma_schedule)}"
          f" — sigma={sigma}, max_gen={n_gen}")
    print("-" * 60)

    result = cmaes_optimize(
        theta_init        = current_theta,
        R1                = R1,
        R2                = R2,
        S0                = S0,
        Kbar              = Kbar,
        theta_baseline    = theta_init,        # always compare vs original LVG
        market_strikes    = market_strikes,    # bid-ask constraint (if loaded)
        bid_prices        = bid_prices,
        ask_prices        = ask_prices,
        initial_step_size = sigma,
        max_generations   = n_gen,
        convergence_tol   = 1e-10,
        random_seed       = 42,
        verbose           = True,
        save_plot         = True,
        plot_path = f'/Users/lucasxia/Desktop/cmaes_stage{stage+1}.png'
    )

    # Best theta from this stage becomes the starting point for the next
    current_theta = result['best_theta']

    print(f"\n  Stage {stage+1} result vs original LVG baseline:")
    print(f"    J           = {result['best_J']:.8f}")
    print(f"    Improvement = {result['improvement']:.2f}%")

# ── Step 3: Multi-Start Refinement ────────────────────────────────────
#
# After the sigma schedule, we are likely in a good local minimum.
# Multi-start perturbs the best solution slightly and runs CMA-ES
# again from each perturbed start, hoping to find an even better basin.
#
# Key settings:
#   - noise = 0.0001: small enough to stay near current best, but large
#     enough to explore slightly different micro-neighborhoods
#   - sigma = 0.0001: very fine search for local refinement
#   - random_seed = trial: different seed each trial = different directions
#   - verbose = False: suppress output since we run 20 trials

print("\n" + "=" * 60)
print("STEP 3: Multi-Start Refinement")
print("=" * 60)
print("Running 20 trials from slightly perturbed starting points...")
print("Each trial uses a different random seed to explore different directions.")

best_J_so_far     = calculate_J(current_theta, R1, R2, S0)[0]
best_theta_so_far = current_theta.copy()

for trial in range(20):
    # Perturb the current best theta by a small amount
    np.random.seed(trial * 7)
    noise           = np.random.randn(len(current_theta)) * 0.0001
    theta_perturbed = current_theta + noise

    result_trial = cmaes_optimize(
        theta_init        = theta_perturbed,
        R1                = R1,
        R2                = R2,
        S0                = S0,
        Kbar              = Kbar,
        theta_baseline    = theta_init,     # still compare vs original LVG
        market_strikes    = market_strikes,
        bid_prices        = bid_prices,
        ask_prices        = ask_prices,
        initial_step_size = 0.0001,
        max_generations   = 1000,
        convergence_tol   = 1e-10,
        random_seed       = trial,          # different seed each trial
        verbose           = False,          # suppress per-generation output
        save_plot         = False,          # skip plots for speed
    )

    improvement_trial = 100 * (J_baseline - result_trial['best_J']) / J_baseline
    print(f"  Trial {trial+1:>2}/20 — J = {result_trial['best_J']:.8f} "
          f"({improvement_trial:.4f}% vs LVG baseline)")

    # Keep track of the best result across all trials
    if result_trial['best_J'] < best_J_so_far:
        best_J_so_far     = result_trial['best_J']
        best_theta_so_far = result_trial['best_theta'].copy()
        print(f"             ↑ New best found!")

current_theta = best_theta_so_far

# ── Final Summary Plot ─────────────────────────────────────────────────
# Generate one final plot comparing the original LVG output vs the
# best theta found across all stages and multi-start trials.

J_final, lam1_final, lam2_final = calculate_J(current_theta, R1, R2, S0)
total_improvement = 100 * (J_baseline - J_final) / J_baseline

print(f"\n{'='*60}")
print(f"FINAL SUMMARY")
print(f"{'='*60}")
print(f"  Original LVG baseline J  = {J_baseline:.8f}")
print(f"  Final optimized J        = {J_final:.8f}")
print(f"  Total improvement        = {total_improvement:.4f}%")
print(f"  Lambda1: {lam1_base:.4f}  ->  {lam1_final:.4f}")
print(f"  Lambda2: {lam2_base:.4f}  ->  {lam2_final:.4f}")
print(f"{'='*60}")

# Save the final comparison plot (baseline LVG vs best optimized result)
_generate_plots(
    history        = {'generation': [0], 'best_J_per_gen': [J_final],
                      'mean_J_per_gen': [J_final], 'step_size': [0],
                      'feasible_count': [1]},
    J_baseline     = J_baseline,
    best_J         = J_final,
    improvement    = total_improvement,
    theta_baseline = theta_init,          # original LVG (always the "before")
    best_theta     = current_theta,        # best found (always the "after")
    R1=R1, R2=R2, S0=S0,
    market_strikes = market_strikes,
    bid_prices     = bid_prices,
    ask_prices     = ask_prices,
    save_path      = '/Users/lucasxia/Desktop/QF Research/optimization results/cmaes_final.png'
)

Loaded 50 market strikes from /Users/lucasxia/Downloads/Quotes.csv
Strike range: 1104 to 1389

Parameter vector length = 208 (expected 2*R1 + 2*R2 = 208)

Original LVG baseline J = 0.00125219
Baseline lambda1 = 0.061297
Baseline lambda2 = 0.140859

Stage 1/7 — sigma=0.3, max_gen=300
------------------------------------------------------------

  Original LVG baseline J  = 0.00125219
  Starting J (this stage)  = 0.00125219
  Lambda1 (init)           = 0.061297
  Lambda2 (init)           = 0.140859
  Bid-ask constraint       = ACTIVE (50 strikes)

  CMA-ES Optimizer  (v4 — direct space + bid-ask constraints)
  Dimension n          = 208
  Population size λ    = 20
  Number of parents μ  = 10
  Effective mass μeff  = 5.94
  Initial step size σ₀ = 0.3
  Max generations      = 300
  Convergence tol      = 1.0e-10
    Gen |         Best J |     Feasible |          σ
  --------------------------------------------------
      0 |     0.00116932 |    19/20    feasible | σ=  0.292179
     20 |  